# 합성 테스트 데이터셋 생성

**왜 합성 테스트 데이터(Synthetic Test Dataset) 인가?**

RAG(검색 증강 생성) 증강 파이프라인의 성능을 평가하는 것은 매우 중요합니다. 

그러나 문서에서 수백 개의 QA(질문-문맥-응답) 샘플을 수동으로 생성하는 것은 시간과 노동력이 많이 소요될 수 있습니다. 또한 사람이 만든 질문은 철저한 평가에 필요한 복잡성 수준에 도달하기 어려워 궁극적으로 평가의 품질에 영향을 미칠 수 있습니다. 

합성 데이터 생성을 사용하면 데이터 집계 프로세스에서 **개발자의 시간을 90%** 까지 줄일 수 있습니다.

In [ ]:
# !pip uninstall -y ragas langchain-community langchain langchain-core langchain-openai
# !pip install "langchain==1.4.2" "langchain-community==0.3.31" "langchain-core==1.6.5" "langchain-openai==1.6.6" "ragas==0.4.3"

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain-community 0.3.31
Uninstalling langchain-community-0.3.31:
  Successfully uninstalled langchain-community-0.3.31
Found existing installation: langchain 1.4.2
Uninstalling langchain-1.4.2:
  Successfully uninstalled langchain-1.4.2
Found existing installation: langchain-core 1.6.5
Uninstalling langchain-core-1.6.5:
  Successfully uninstalled langchain-core-1.6.5
Found existing installation: langchain-openai 1.6.6
Uninstalling langchain-openai-1.6.6:
  Successfully uninstalled langchain-openai-1.6.6
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.6.5-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_openai-1.6.6-py3-none-any.whl.metadata (3.4 kB)
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
Using


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import langchain_community, ragas
print(langchain_community.__version__)
print(ragas.__version__)

d:\hykim\rag_two_branch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.3.31
0.4.3


In [4]:
import sys
print(sys.executable)
import langchain_community, ragas, langchain_openai
print(langchain_community.__version__)
print(ragas.__version__)
print(langchain_openai.__version__)

d:\hykim\rag_two_branch\.venv\Scripts\python.exe
0.3.31
0.4.3
1.6.6


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

## 문서 전처리

In [6]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

각 문서 객체에는 `metadata` 를 통해 액세스할 수 있는 문서에 대한 추가 정보를 저장하는 데 사용할 수 있는 메타데이터 사전이 포함되어 있습니다. 

메타데이터 사전에는 `filename` 이라는 키가 포함되어 있는지 확인하세요. 

이 키는 Test datasets 생성 프로세스에서 활용될 것이므로. 메타데이터의 `filename` 속성은 동일한 문서에 속한 청크를 식별하는 데 사용됩니다. 

In [7]:
docs[0].metadata

{'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [8]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

## 데이터셋 생성

In [14]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터셋 생성 모델
generator_llm = ChatOpenAI(model="gpt-4o-mini")
# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Ragas 0.4.x에서는 KeyphraseExtractor와 critic_llm을 직접 구성하지 않습니다.
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [15]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

# 페이지를 분할하지 않고 각 DOCUMENT에 필요한 정보를 추출
transforms = [       

    SummaryExtractor(llm=generator_llm),      # LLM으로 문서 내용을 요약

    EmbeddingExtractor(                       # 지정한 텍스트 속성을 숫자 벡터로 변환
        embedding_model=generator.embedding_model,
        property_name="summary_embedding",
        embed_property_name="summary",
    ),

    NERExtractor(llm=generator.llm),          # 개체명 인식(Named Entity Recognition)으로 사람·기관·장소 등 주요 개체를 추출
]

# 한 페이지의 개체 정보를 바탕으로 질문 생성
query_distribution = [      
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),   # 하나의 문맥에서 답할 수 있는 구체적인 질문과 참조 정답을 생성
]

In [16]:
# 테스트셋 생성
testset = generator.generate_with_langchain_docs(
    documents=docs,           # PDF 문서에서 추출한 페이지를 DOCUMENT로 변환하여 전달
    testset_size=10,          # 생성할 테스트셋의 질문 수
    transforms=transforms,    # 각 DOCUMENT에 적용할 변환기(요약, 임베딩, 개체명 인식)
    query_distribution=query_distribution,    # 한 페이지의 개체 정보를 바탕으로 질문 생성
    raise_exceptions=True,    # 질문 생성 중 오류 발생 시 예외를 발생시키도록 설정
)

Generating Samples: 100%|██████████| 10/10 [00:02<00:00,  3.96it/s]
